In [ ]:
#| default_exp dashboard

In [ ]:
#| export
"""Pure workflow helpers used by the Streamlit dashboard."""
from __future__ import annotations

import ast
from io import BytesIO
import json
from pathlib import Path
import re
import tempfile
import zipfile

import pandas as pd

from probeharbour.facade import ProbeStore

CART_CSV_SCHEMA_VERSION = 2


def load_cart_csv(source, *, name: str = "imported_probe_cart") -> ProbeStore:
    """Load a ProbeHarbour export or sequence-only 10x add-on CSV as a cart."""
    frame = pd.read_csv(source)
    normalized_columns = {str(column).strip().lower(): column for column in frame.columns}
    aliases = {
        "name": "probe_name",
        "rbdm5": "rbd5",
        "rbdm3": "rbd3",
    }
    rename = {
        normalized_columns[source_name]: target_name
        for source_name, target_name in aliases.items()
        if source_name in normalized_columns and target_name not in frame
    }
    frame = frame.rename(columns=rename)
    missing = {"rbd5", "rbd3"} - set(frame)
    if missing:
        raise ValueError(f"Cart CSV is missing required columns: {sorted(missing)}")
    if "candidate_id" not in frame:
        if "probe_name" not in frame:
            raise ValueError("Cart CSV requires candidate_id or probe_name")
        frame["candidate_id"] = frame["probe_name"].astype(str)

    structured_columns = {
        "coordinates", "provenance", "characterization", "specificity_evidence", "extra",
        "probeharbour_score_modifiers", "score_modifiers",
        "panel_compatibility_reasons", "existing_panel_compatibility_reasons",
        "junction_supporting_transcripts", "supporting_transcripts",
    }

    def restore(value):
        if not isinstance(value, str) or not value.strip():
            return value
        text = value.strip()
        if text.lower() in {"nan", "none", "<na>"}:
            return None
        try:
            return json.loads(text)
        except (json.JSONDecodeError, TypeError):
            try:
                return ast.literal_eval(text)
            except (SyntaxError, ValueError):
                return value

    for column in structured_columns.intersection(frame.columns):
        frame[column] = frame[column].map(restore)
    frame = frame.drop(columns=["panel_record_id"], errors="ignore")
    frame = frame.drop_duplicates("candidate_id", keep="last").reset_index(drop=True)
    if "gene_name" not in frame:
        frame["gene_name"] = pd.NA
    for alias in ("gene", "gene_id"):
        if alias in frame:
            frame["gene_name"] = frame["gene_name"].fillna(frame[alias])
    if "probe_name" in frame:
        inferred_gene = frame["probe_name"].astype(str).str.split("_", n=1).str[0]
        frame["gene_name"] = frame["gene_name"].fillna(inferred_gene)
    frame["panel_role"] = "add_on"
    frame["selection_status"] = "imported into dashboard cart"
    return ProbeStore(frame, name=name, panel_role="add_on")


def junction_summary(data) -> pd.DataFrame:
    """Return one auditable row per designed junction for UI selection."""
    frame = getattr(data, "obs", data)
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("data must be a ProbeStore or pandas DataFrame")
    if "junction_id" not in frame:
        raise ValueError("Designed candidates require a junction_id column")
    columns = {
        "gene_name": "gene",
        "junction_specificity_class": "junction_class",
        "junction_transcript_count": "transcripts_with_junction",
        "gene_transcript_count": "annotated_transcripts",
        "target_coverage": "transcript_coverage_percent",
        "junction_supporting_transcripts": "supporting_transcripts",
    }
    aggregations = {target: (source, "first") for source, target in columns.items() if source in frame}
    aggregations["candidate_count"] = ("junction_id", "size")
    result = frame.groupby("junction_id", as_index=False).agg(**aggregations)
    coordinates = result["junction_id"].str.extract(r"^(?:chr)?([^:]+):(\d+)-(\d+):([+-])$")
    result["_chrom"] = coordinates[0].fillna("")
    result["_start"] = pd.to_numeric(coordinates[1], errors="coerce").fillna(float("inf"))
    result = result.sort_values(["_chrom", "_start", "junction_id"]).reset_index(drop=True)
    result.insert(0, "junction_label", [f"J{index + 1}" for index in range(len(result))])
    preferred = [
        "junction_label", "junction_id", "gene", "junction_class", "transcripts_with_junction",
        "annotated_transcripts", "transcript_coverage_percent", "supporting_transcripts",
        "candidate_count",
    ]
    return result[[column for column in preferred if column in result]].reset_index(drop=True)


def candidate_review_table(data, *, score_column: str = "probeharbour_score") -> pd.DataFrame:
    """Compact review table with stable columns for Streamlit's data editor."""
    frame = getattr(data, "obs", data).copy()
    required = {"candidate_id", "junction_id", "rbd5", "rbd3", score_column}
    missing = required - set(frame)
    if missing:
        raise ValueError(f"Candidate review requires columns: {sorted(missing)}")
    specificity = next((name for name in ("specificity_percent_recalculated", "specificity_score")
                        if name in frame), None)
    sort_columns = ["junction_id", score_column] + ([specificity] if specificity else []) + ["candidate_id"]
    ascending = [True, False] + ([False] if specificity else []) + [True]
    frame = frame.sort_values(sort_columns, ascending=ascending)
    frame["junction_rank"] = frame.groupby("junction_id").cumcount() + 1
    frame["include"] = frame["junction_rank"].eq(1)
    columns = [
        "include", "junction_rank", "candidate_id", "probe_name", "gene_name",
        "junction_id", "shift_nt", "lig_dinuc", "rbd5", "rbd3", "tm_full",
        score_column, f"{score_column}_modifiers", specificity,
        "existing_panel_compatibility_pass", "panel_compatibility_pass",
    ]
    return frame[[column for column in columns if column and column in frame]].reset_index(drop=True)


def selected_panel(data, candidate_ids, *, name: str = "custom_isoform_panel") -> ProbeStore:
    """Build a selected ProbeStore while retaining complete candidate evidence."""
    frame = getattr(data, "obs", data)
    wanted = {str(value) for value in candidate_ids}
    chosen = frame.loc[frame["candidate_id"].astype(str).isin(wanted)].copy()
    missing = wanted - set(chosen["candidate_id"].astype(str))
    if missing:
        raise KeyError(f"Unknown candidate IDs: {sorted(missing)}")
    if chosen.empty:
        raise ValueError("Select at least one candidate")
    chosen["probe_name"] = chosen.get("probe_name", chosen["candidate_id"])
    chosen["selection_status"] = "confirmed in dashboard"
    chosen["panel_role"] = "add_on"
    return ProbeStore(chosen.reset_index(drop=True), name=name, panel_role="add_on")


def add_to_basket(basket, additions, *, name: str = "xenium_add_on_panel") -> ProbeStore:
    """Add confirmed candidates to a persistent ProbeSet, deduplicating by ID."""
    addition_frame = getattr(additions, "obs", additions).copy()
    if "candidate_id" not in addition_frame:
        raise ValueError("Basket additions require a candidate_id column")
    frames = [] if basket is None else [getattr(basket, "obs", basket).copy()]
    frames.append(addition_frame)
    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined = combined.drop_duplicates("candidate_id", keep="last").reset_index(drop=True)
    # Row IDs are local to the panel on which compatibility was last run. They
    # must be regenerated after combining independently audited gene panels.
    combined = combined.drop(columns=["panel_record_id"], errors="ignore")
    if "gene_name" not in combined:
        combined["gene_name"] = pd.NA
    for alias in ("gene", "gene_id"):
        if alias in combined:
            combined["gene_name"] = combined["gene_name"].fillna(combined[alias])
    combined["panel_role"] = "add_on"
    combined["selection_status"] = "confirmed in dashboard basket"
    return ProbeStore(combined, name=name, panel_role="add_on")


def remove_from_basket(basket, candidate_ids, *, name: str | None = None) -> ProbeStore:
    """Remove candidates from a ProbeSet basket while preserving remaining evidence."""
    frame = getattr(basket, "obs", basket).copy()
    if "candidate_id" not in frame:
        raise ValueError("Basket requires a candidate_id column")
    remove = {str(value) for value in candidate_ids}
    remaining = frame.loc[~frame["candidate_id"].astype(str).isin(remove)].reset_index(drop=True)
    basket_name = name or getattr(basket, "name", "xenium_add_on_panel")
    return ProbeStore(remaining, name=basket_name, panel_role="add_on")


def export_panel_bundle(store: ProbeStore, *, panel_name: str | None = None) -> bytes:
    """Return a ZIP containing portable ProbeStore, CSV, BED and FASTA exports."""
    name = panel_name or store.name
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", str(name)).strip("._") or "probe_panel"
    with tempfile.TemporaryDirectory(prefix="probestore-dashboard-") as directory:
        root = Path(directory) / safe_name
        root.mkdir()
        store.write_csv(root / f"{safe_name}.csv")
        store.write_bed(root / f"{safe_name}.bed")
        store.write_fasta(root / f"{safe_name}.fasta")
        portable = root / "probestore"
        store.save(portable)
        manifest = {
            "panel_name": safe_name,
            "n_probes": len(store.obs),
            "candidate_ids": store.obs["candidate_id"].astype(str).tolist(),
            "files": [f"{safe_name}.csv", f"{safe_name}.bed", f"{safe_name}.fasta",
                      "probestore/probes.parquet", "probestore/metadata.json"],
        }
        (root / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
        buffer = BytesIO()
        with zipfile.ZipFile(buffer, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(root.rglob("*")):
                if path.is_file():
                    archive.write(path, Path(safe_name) / path.relative_to(root))
        return buffer.getvalue()